# RepoScan Vehicle Attribute VLM Labeling

Use this notebook in Google Colab when the local machine does not have a GPU. It labels the vehicle crop review CSV with Qwen2.5-VL, then creates a pending AI-assisted consensus CSV for warm-start training. Pending consensus is not a replacement for approved human review.

In [ ]:
REPO_URL = "https://github.com/MarkW1919/reposcan-pro.git"
BRANCH = "codex/bootstrap"
WORKDIR = "/content/reposcan-pro"

# Upload or mount these before running the labeling cells.
REVIEW_CSV = "/content/open_images_vehicle_attribute_crop_suggestions_20260415.csv"
CROP_ROOT = "/content/open_images_vehicle_attribute_review_20260415/crops"

COLAB_REVIEW_CSV = "/content/open_images_vehicle_attribute_crop_suggestions_colab_paths.csv"
VLM_OUTPUT_CSV = "/content/open_images_vehicle_attribute_crop_vlm_suggestions_20260415.csv"
CONSENSUS_OUTPUT_CSV = "/content/open_images_vehicle_make_model_ai_consensus_pending_20260415.csv"

# Start with a smaller batch, then set to None for a full run.
LIMIT = 200

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!python -m pip install -q -e ".[vlm-labeling]"

Upload the crop CSV and a zip/folder containing the crop images. If the CSV has Windows paths, the next cell rewrites `crop_filepath` to point at `CROP_ROOT` by filename.

In [ ]:
from google.colab import files

# Optional uploader. You can skip this cell if files are already in /content or mounted from Drive.
uploaded = files.upload()
uploaded

In [ ]:
from pathlib import Path
import csv

review_path = Path(REVIEW_CSV)
crop_root = Path(CROP_ROOT)
rewritten_path = Path(COLAB_REVIEW_CSV)

with review_path.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    rows = [dict(row) for row in reader]
    fieldnames = list(reader.fieldnames or [])

missing = []
for row in rows:
    original = str(row.get("crop_filepath") or "").replace("\\", "/")
    candidate = crop_root / Path(original).name
    row["crop_filepath"] = str(candidate)
    if not candidate.exists():
        missing.append(str(candidate))

with rewritten_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"rewrote {len(rows)} rows -> {rewritten_path}")
print(f"missing crop files: {len(missing)}")
missing[:5]

In [ ]:
limit_args = "" if LIMIT is None else f"--limit {LIMIT}"
!python scripts/label_vehicle_attribute_crops_with_vlm.py label-crops \
  --review-csv {COLAB_REVIEW_CSV} \
  --output-csv {VLM_OUTPUT_CSV} \
  --provider qwen2_5_vl \
  --model Qwen/Qwen2.5-VL-7B-Instruct \
  --overwrite {limit_args}

In [ ]:
!python scripts/label_vehicle_attribute_crops_with_vlm.py apply-consensus \
  --input-csv {VLM_OUTPUT_CSV} \
  --output-csv {CONSENSUS_OUTPUT_CSV} \
  --task vehicle_make_model_classification \
  --min-vlm-confidence 0.75 \
  --overwrite

In [ ]:
!zip -j /content/reposcan_vlm_label_outputs.zip {VLM_OUTPUT_CSV} {CONSENSUS_OUTPUT_CSV}
files.download("/content/reposcan_vlm_label_outputs.zip")